In [ ]:
from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
import pandas as pd

In [22]:
df = pd.read_csv('../data/preprocessed/crypto_data_proc.csv')
print(f'Total rows: {len(df):,}')
print(f'Tokens: {df["token"].nunique()}')
print(f'Time range: {df["time_idx"].min()} to {df["time_idx"].max()}')
df.head()

Total rows: 1,011,769
Tokens: 30
Time range: 0 to 52029


,time_idx,token,open_log,high_log,low_log,close_log,volume_log,btc_close_log,eth_close_log,eth_btc_ratio,hour_sin,hour_cos,norm_day_sin,norm_day_cos,weekday_sin,weekday_cos,month_sin,month_cos,year,next_close_log
0,0,1INCH,2.558157,-0.138144,2.409150,-0.030149,16.926395,0.001111,0.007636,0.025986,1.000000,6.123234e-17,-0.937752,0.347305,-0.433884,-0.900969,-2.449294e-16,1.0,2020,0.045938
1,1,1INCH,-0.026367,0.070677,0.059331,0.045938,16.874238,0.001031,-0.001646,0.025916,0.965926,-2.588190e-01,-0.937752,0.347305,-0.433884,-0.900969,-2.449294e-16,1.0,2020,-0.003933
2,2,1INCH,0.045316,-0.021816,0.043149,-0.003933,16.677244,0.001167,0.006651,0.026059,0.866025,-5.000000e-01,-0.937752,0.347305,-0.433884,-0.900969,-2.449294e-16,1.0,2020,0.008800
3,3,1INCH,-0.008165,-0.026874,0.038948,0.008800,16.110003,0.010799,0.010399,0.026049,0.707107,-7.071068e-01,-0.937752,0.347305,-0.433884,-0.900969,-2.449294e-16,1.0,2020,0.048203
4,4,1INCH,0.009949,0.079119,0.027554,0.048203,16.923341,0.001701,0.003810,0.026104,0.500000,-8.660254e-01,-0.937752,0.347305,-0.433884,-0.900969,-2.449294e-16,1.0,2020,-0.044002


# Train/Val/Test Split (Per Token)
Since each token has its own time series with different lengths, we split each token's data individually using the same percentage thresholds.

In [ ]:
def split_by_token(group, train_pct=0.70, val_pct=0.15):
  """
  Split time series sequentially to prevent data leakage.
  Train: earliest 70% of data
  Val: next 15% of data
  Test: latest 15% of data

  This mimics production: training on past, predicting future.
  """
  max_idx = group['time_idx'].max()
  train_cutoff = int(train_pct * max_idx)
  val_cutoff = int((train_pct + val_pct) * max_idx)

  group['split'] = 'test'  # default
  group.loc[group['time_idx'] <= train_cutoff, 'split'] = 'train'
  group.loc[
    (group['time_idx'] > train_cutoff) & (group['time_idx'] <= val_cutoff), 'split'
  ] = 'val'

  return group


# Apply split to each token
df = df.groupby('token', group_keys=False).apply(split_by_token)

# Verify splits
print('Split percentages:')
print(df['split'].value_counts(normalize=True) * 100)

Split percentages:
split
train    69.999773
test     15.000756
val      14.999471
Name: proportion, dtype: float64


/var/folders/zf/pd74vy996kg6l6dpbrpqm1jr0000gn/T/ipykernel_60576/1870572874.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('token', group_keys=False).apply(split_by_token)


In [24]:
# Create separate dataframes for each split
train_df = df[df['split'] == 'train'].copy().drop(columns=['split'])
val_df = df[df['split'] == 'val'].copy().drop(columns=['split'])
test_df = df[df['split'] == 'test'].copy().drop(columns=['split'])

print(f"Train: {len(train_df):,} rows")
print(f"Val: {len(val_df):,} rows")
print(f"Test: {len(test_df):,} rows")

Train: 708,236 rows
Val: 151,760 rows
Test: 151,773 rows


# Create TimeSeriesDataSet with RobustGroupNormalizer

In [ ]:
# Define parameters
max_prediction_length = 1  # Predict 1 step ahead (next_close_log)
min_encoder_length = 168  # 1 week of hourly data
max_encoder_length = 168  

# Financial features that need RobustGroupNormalizer (log returns)
financial_features = [
  'open_log',
  'high_log',
  'low_log',
  'close_log',
  'volume_log',
  'btc_close_log',
  'eth_close_log',
  'eth_btc_ratio',
]

# Cyclical temporal features (already bounded [-1, 1], no normalization needed)
temporal_features = [
  'hour_sin',
  'hour_cos',
  'norm_day_sin',
  'norm_day_cos',
  'weekday_sin',
  'weekday_cos',
  'month_sin',
  'month_cos',
  'year'
]

# Create scalers dictionary for financial features
upper_q = 0.95
lower_q = 0.05
scalers = {
  # Price log features - use ROBUST scaling due to outliers (especially low_log)
  'open_log': GroupNormalizer(
    groups=['token'],
    method='robust',
    center=True,
    transformation=None,
    method_kwargs={'upper': upper_q, 'lower': lower_q},
  ),
  'high_log': GroupNormalizer(
    groups=['token'],
    method='robust',
    center=True,
    transformation=None,
    method_kwargs={'upper': upper_q, 'lower': lower_q},
  ),
  'low_log': GroupNormalizer(
    groups=['token'],
    method='robust',
    center=True,
    transformation=None,
    method_kwargs={'upper': upper_q, 'lower': lower_q},
  ),
  'close_log': GroupNormalizer(
    groups=['token'],
    method='robust',
    center=True,
    transformation=None,
    method_kwargs={'upper': upper_q, 'lower': lower_q},
  ),
  # Volume: use ROBUST scaling (skewed distribution, mean ≈> max suggests heavy skew)
  'volume_log': GroupNormalizer(
    groups=['token'],
    method='robust',
    center=True,
    transformation=None,
    method_kwargs={'upper': upper_q, 'lower': lower_q},
  ),
  # BTC/ETH log returns - smaller ranges, but robust is still safer for crypto
  'btc_close_log': GroupNormalizer(
    groups=[],  # we want to normalize relative to itself not each token
    method='robust',
    center=True,
    transformation=None,
    method_kwargs={'upper': upper_q, 'lower': lower_q},
  ),
  'eth_close_log': GroupNormalizer(
    groups=[],  # we want to normalize relative to itself not each token
    method='robust',
    center=True,
    transformation=None,
    method_kwargs={'upper': upper_q, 'lower': lower_q},
  ),
  # ETH/BTC ratio - bimodal! But it's a regime signal so preserve it
  'eth_btc_ratio': GroupNormalizer(
    groups=[],  # we want to normalize relative to itself not each token
    method='robust',
    center=True,
    transformation=None,
    method_kwargs={'upper': upper_q, 'lower': lower_q},
  ),
}


print(f'Encoder length: {max_encoder_length} timesteps')
print(f'Prediction length: {max_prediction_length} timesteps')
print(f'Financial features with RobustGroupNormalizer: {len(financial_features)}')

Encoder length: 168 timesteps
Prediction length: 1 timesteps
Financial features with RobustGroupNormalizer: 8


In [ ]:
training = TimeSeriesDataSet(
  train_df,
  time_idx='time_idx',
  target='next_close_log',
  group_ids=['token'],
  # Sequence lengths
  max_encoder_length=max_encoder_length,
  max_prediction_length=max_prediction_length,
  min_encoder_length=min_encoder_length,
  # Known features (available at prediction time)
  time_varying_known_reals=temporal_features,
  # Unknown features (only known historically, including all financial features)
  time_varying_unknown_reals=financial_features,
  # Categorical features
  static_categoricals=['token'],
  # Target normalization
  target_normalizer=GroupNormalizer(
    groups=['token'],  # we want to normalize relative to itself not each token
    method='robust',
    center=True,
    transformation=None,
    method_kwargs={'upper': upper_q, 'lower': lower_q},
  ),
  # Apply RobustGroupNormalizer to all financial features
  scalers=scalers,
  # Additional parameters
  add_relative_time_idx=True,
  add_target_scales=True,
  add_encoder_length=False,
  allow_missing_timesteps=False,
  randomize_length=False
)

print('Training dataset created successfully!')
print(f'Number of samples: {len(training)}')

Training dataset created successfully!
Number of samples: 711836


In [ ]:
validation = TimeSeriesDataSet.from_dataset(
  training,
  val_df,
  stop_randomization=True
)

print("Validation dataset created successfully!")
print(f"Number of samples: {len(validation)}")

Validation dataset created successfully!
Number of samples: 155360


In [ ]:
test = TimeSeriesDataSet.from_dataset(
  training,
  test_df,
  predict=False,
  stop_randomization=True
)

print("Test dataset created successfully!")
print(f"Number of samples: {len(test)}")

Test dataset created successfully!
Number of samples: 155373


In [ ]:
# Save the datasets
training.save('../data/training_temporaldataset.pkl')
validation.save('../data/validation_temporaldataset.pkl')
test.save('../data/test_temporaldataset.pkl')

"""
To load:
from pytorch_forecasting import TimeSeriesDataSet

# Load the datasets
training = TimeSeriesDataSet.load('training_dataset.pkl')
validation = TimeSeriesDataSet.load('validation_dataset.pkl')
test = TimeSeriesDataSet.load('test_dataset.pkl')
"""

"\nTo load:\nfrom pytorch_forecasting import TimeSeriesDataSet\n\n# Load the datasets\ntraining = TimeSeriesDataSet.load('training_dataset.pkl')\nvalidation = TimeSeriesDataSet.load('validation_dataset.pkl')\ntest = TimeSeriesDataSet.load('test_dataset.pkl')\n"